# Class 20: Hypothesis tests and confidence intervals

Plan for today. Each numbered section below follows a part of the slides, so when you reach the next section, go back to the slides.

1. Hypothesis tests for correlation
2. Visual hypothesis tests
3. Two-sided hypothesis tests
4. Confidence intervals

The hypothesis test for two means (smoking and baby weights) is in the class 19 notebook, section 3.

In [ ]:
import YData

# YData.download.download_class_code(20)   # get class code    
# YData.download.download_class_code(20, True) # get the code with the answers 

YData.download_data("amazon.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline


## 1. Hypothesis tests for correlation

Let's run a hypothesis test for correlation to see if books that have more pages cost more! 


### Step 1: State the null and alternative hypotheses

**In words** 

Null hypothesis: There is no correlation between the number of pages in a book and the cost of the book. 

Alternative hypothesis: Books that have more pages cost more. 

**In symbols**

$H_0$: $\rho = 0$  

$H_A$: $\rho > 0$  


### Step 2: Calculate the observed statistic

Data on 230 books from Amazon.com are loaded below. Let's calculate the observed correlation ($r$) between the number of pages in the book (`NumPages`) and the listed price (`List.Price`). 

To make your life easier, first just reduce the data set to only the `NumPages` and `List.Price` columns and save this to a DataFrame called `amazon_smaller`. Then use `amazon_smaller` for the rest of this problem


In [ ]:
amazon = pd.read_csv("amazon.csv")

amazon.head(3)

In [ ]:
import statistics

amazon_smaller = amazon[["NumPages", "List.Price"]]

obs_stat = ...

obs_stat


### Step 3: Create the null distribution 

How can we create one statistic consistent with the null hypothesis? 

See if you can create one statistic consistent with the null distribution below. Once you have done that, create the full null distribution!



In [ ]:
# Create one statistic consistent with the null distribution
...


In [ ]:
%%time

# Create the full null distribution

null_dist = []

for i in range(10000):  
        
    ...



In [ ]:
# visualize the null distribution 

plt.hist(null_dist, edgecolor = "black", bins = 100);


# put a line at the observed statistic value

plt.axvline(obs_stat, color = "red");
plt.xlabel("r");
plt.ylabel("Count");

### Step 4: Calculate the p-value


In [ ]:
p_value = ...

p_value


### Step 5: Draw a conclusion

The p-value is small (less than the conventional level of 0.05), so we would reject the null hypothesis and conclude that there is a positive correlation between the number of pages in a book and the price of a book at the population level. 


## 2. Visualizing hypothesis tests for correlation

We can also run a visual hypothesis test for correlation by creating a visual lineup that displays several scatter plots of shuffled data and one scatter plot of the real data. If you can tell which plot contains the real (unshuffled) data, this corresponds to being able to reject the null hypothesis. 


In [ ]:
# np.random.seed(123)

def create_lineup(the_data): 
    
    real_plot_num = np.random.permutation(range(20))[0]
    lineup_data = pd.DataFrame()
    
    for i in range(20):
        
        if i == real_plot_num:
            curr_data = the_data.copy()
        
        else:
            curr_data = pd.DataFrame(the_data.iloc[:, 1], the_data.iloc[:, 0]).reset_index()
            curr_data[curr_data.columns[1]] = np.random.permutation(curr_data[curr_data.columns[1]])
        
        curr_data.loc[:, "Lineup"] = i
        lineup_data = pd.concat([lineup_data, curr_data])
        
    lineup_data = lineup_data.reset_index()

    sns.set_context("paper", rc={"axes.labelsize":20})   
    sns.relplot(lineup_data, x = lineup_data.columns[1], y = lineup_data.columns[2], col = "Lineup", col_wrap=5); 

    return real_plot_num


the_answer = create_lineup(amazon_smaller)



In [ ]:
# display the answer

# the_answer


## 3. Two-sided hypothesis test

Sometimes in hypothesis testing we don't know the direction of an effect; we only know that the null hypothesis is incorrect. 

In these circumstances, we write our alternative hypothesis such that we state that the parameter value is not equal to the value specified by the null hypothesis.

For testing whether there is a correlation between the number of pages in a book and its price, we can write our hypotheses as:

$H_0: \rho = 0$   

i.e., the null hypothesis is the same as before.

$H_A: \rho \ne 0$

We now use not equal to ($\ne$) in our alternative hypothesis.  

To calculate the p-value, we need to look at the values more extreme than the observed statistic in both tails. 

In [ ]:
# visualize the null distribution 

plt.hist(null_dist, edgecolor = "black", bins = 100);


# put lines showing values more extreme than the observed statistic

plt.axvline(obs_stat, color = "red");
plt.axvline(-1 * obs_stat, color = "red");

plt.xlabel("Null statistic value");
plt.ylabel("Count");


When calculating the p-value, we need to get the proportion of statistics in the null distribution that are more extreme than the observed statistic from both tails.

In [ ]:
p_value_right = np.mean(np.array(null_dist) >= obs_stat)
p_value_left = np.mean(np.array(null_dist) <= -1 * obs_stat)

p_value = p_value_right + p_value_left

p_value 


## 4. Using hypothesis tests to generate confidence intervals

There are several methods that can be used to calculate confidence intervals, including using a computational method called the "bootstrap" and using "parametric methods" that involve using probability distributions. If you take a traditional introductory statistics class you will learn some of these methods.

Below we use a less conventional method to calculate confidence intervals by looking at all parameter values that a hypothesis test fails to reject (at the p-value < 0.05 level). As you will see, the method gives similar results to other methods, although it requires a bit more computation time.

As an example, let's create a confidence interval for the population proportion of movies $\pi$ that pass the Bechdel test. As is the case for all confidence intervals, this confidence interval gives a range of plausible values that likely contains the true population proportion $\pi$.


In [ ]:
# To start, let's use a function that generates a statistic p-hat that is consistent with a particular population parameter value pi

def generate_proportion(n, prob_heads):
    
    random_sample = np.random.rand(n) <= prob_heads
    return np.mean(random_sample)

generate_proportion(1794, .5)



In [ ]:
# The function below calculates a p-value for the Bechdel data based on a particular pi value that is specified in a null hypothesis.
# (i.e., it is a function that encapsulates the hypothesis test you ran in class 17).


def get_Bechdel_pvalue(null_hypothesis_pi, plot_null_dist = False):
    
    
    # The observed p-hat value
    prop_passed = 803/1794
    
    
    # Generate the null distribution 
    null_dist = []
    
    for i in range(10000):    
        null_dist.append(generate_proportion(1794, null_hypothesis_pi))
    
    
    # Calculate a "two-tailed" p-value which is the proportion of statistcs more extreme than the observed statistic

    statistic_deviation = np.abs(null_hypothesis_pi - prop_passed)  # distance between p-hat and pi_0
    
    pval_left = np.mean(np.array(null_dist) <= null_hypothesis_pi - statistic_deviation)
    pval_right = np.mean(np.array(null_dist) >= null_hypothesis_pi + statistic_deviation)    
    
    p_value = pval_left + pval_right


    
    # plot the null distribution and lines indicating values more extreme than the observed statistic 
    if plot_null_dist:
        
        plt.hist(null_dist, edgecolor = "black", bins = 30);
        plt.axvline(null_hypothesis_pi - statistic_deviation, color = "red");
        plt.axvline(null_hypothesis_pi + statistic_deviation, color = "red");
        plt.axvline(prop_passed, color = "green");
        plt.axvline(null_hypothesis_pi, color = "yellow");

        
        plt.title("Pi-null is: " + str(null_hypothesis_pi) + "      "  +
                  "p-value is: " + str(round(p_value, 5)))
      
    # return the p-value
    return p_value
    


In [ ]:
# test the function with the value H0: pi = .5  (as we did in class 17)


# test the function with the value H0: pi = .45


In [ ]:
# create a range of H0: pi = x  values

possible_null_pis = ...

possible_null_pis    


In [ ]:
%%time


# get the p-value for a range of H0: pi = x  values


pvalues = []

...



In [ ]:
# view the p-values 
# convention calls a p-value < 0.05 "statistically significant", indicating a pi incompatible with the null hypothesis
# our confidence interval is all pi values that are not statistically significant (i.e., pi values that are consistent with particular H0)

pvalue_df = ...

pvalue_df


In [ ]:
# plot p-values as a function of H0 pi's

sns.relplot(pvalue_df, x = 'pi', y = 'p-values');
plt.xticks(rotation=90);
plt.axhline(.05, color = "red");

plt.plot(pvalue_df['pi'], pvalue_df['non-significant'], color = "green");


In [ ]:
# Get all plausible Pi values
fail_to_reject_pis = ...

fail_to_reject_pis


In [ ]:
# get the CI as the max and min plausible pi values 

...


In [ ]:
# using the statsmodels package to compute a confidence interval for a proportion

import statsmodels.api as sm

ci_low, ci_upp = sm.stats.proportion_confint(803, 1794, alpha=0.05, method='normal')
(round(ci_low, 3), round(ci_upp, 3))
